# Cohere Jupyter GUI

Bragg CDI reconstruction in the notebook, an alternative to the PyQt5 `cohere_gui`.

```bash
pip install cohere-ui[jupyter]
```

In [ ]:
from cohere_ui.jupyter_gui import CoherenceGUI

gui = CoherenceGUI()
gui.display()

**Load an experiment**: browse in the picker above, or set `EXP_DIR` and run:

In [ ]:
import os

EXP_DIR = os.environ.get('EXP_DIR', '')   # or inline, e.g. '/data/scan_54'
if EXP_DIR:
    gui.load_experiment(EXP_DIR)
    print('loaded:', gui.results.experiment_dir, '| tabs:', list(gui._tabs))
else:
    print('Browse in the picker above, or set EXP_DIR.')

**External viewers** (optional): point the Prep-tab *ImageJ* and Postprocess *ParaView* buttons at your installs. Skip if auto-detection already finds them.

In [ ]:
import cohere_ui.jupyter_gui as cgui

# Uncomment + edit only if the buttons can't auto-detect your install:
# cgui.IMAGEJ_PATH   = '/Applications/ImageJ.app'      # Prep-tab TIFF buttons
# cgui.PARAVIEW_PATH = '/Applications/ParaView.app'    # Postprocess viewer
# (The IMAGEJ / PARAVIEW environment variables work too.)
print('IMAGEJ_PATH =', cgui.IMAGEJ_PATH, '| PARAVIEW_PATH =', cgui.PARAVIEW_PATH)

**Inspect results** (after a run completes): arrays live on `gui.results`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gui.results.reload()
image, support, errors = gui.results.image, gui.results.support, gui.results.errors

if image is None:
    print('No results yet, run a reconstruction or load an experiment first.')
else:
    mid = image.shape[0] // 2
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(np.abs(image[mid]));                    ax[0].set_title('amplitude')
    ax[1].imshow(np.angle(image[mid]), cmap='twilight'); ax[1].set_title('phase')
    if support is not None:
        ax[2].imshow(support[mid]);                      ax[2].set_title('support')
    plt.tight_layout(); plt.show()
    print(f'image {image.shape} {image.dtype}')
    if errors is not None:
        errs = np.asarray(errors).ravel()
        plt.figure(figsize=(8, 3))
        plt.semilogy(errs); plt.grid(True, alpha=0.3)
        plt.xlabel('iteration'); plt.ylabel('error')
        plt.title(f'{errs.size} iters, final {errs[-1]:.4g}'); plt.show()

**Interactive 3D** (real space): rotate, zoom, and read phase on the amplitude isosurface.

In [ ]:
import numpy as np
import pyvista as pv
from IPython.display import display
from ipywidgets import HTML

# Render the scene in the browser (client-side WebGL). Do NOT use the default
# 'trame'/'server' backends here: on macOS they render VTK on a background
# thread, which aborts the kernel (NSException, no Python traceback).
pv.set_jupyter_backend('client')

gui.results.reload()
image = gui.results.image

if image is None:
    print('No results yet, run a reconstruction first.')
else:
    amp = np.abs(image).astype(np.float32)
    peak = float(amp.max())

    grid = pv.ImageData()
    grid.dimensions = np.array(amp.shape) + 1
    grid.cell_data['amplitude'] = amp.flatten(order='F')
    grid.cell_data['phase'] = np.angle(image).astype(np.float32).flatten(order='F')
    grid = grid.cell_data_to_point_data()

    display(HTML(f'<b>image</b> {image.shape} {image.dtype}, peak {peak:.4g}'))

    pl = pv.Plotter(notebook=True, window_size=(720, 540))
    pl.set_background('white'); pl.add_axes()
    iso = grid.contour([peak * 0.3], scalars='amplitude')
    pl.add_mesh(iso, scalars='phase', cmap='twilight', clim=(-np.pi, np.pi))
    pl.show()

**Reciprocal-space check**:s measured diffraction vs `|FFT(image)|^2` (log-scaled isosurfaces). A converged reconstruction overlaps the measurement.

In [ ]:
import numpy as np
import pyvista as pv

# Client-side WebGL render — see the note in the 3D cell.
pv.set_jupyter_backend('client')

gui.results.reload()
image, data = gui.results.image, gui.results.data

if image is None or data is None:
    print('Need both image and data, run a reconstruction first.')
else:
    recon = np.fft.fftshift(np.abs(np.fft.fftn(image)) ** 2)
    if recon.shape != data.shape:            # image (x,y,z) FFT vs data (z,y,x)
        recon = np.transpose(recon, (2, 1, 0))

    def log_iso(vol, frac=0.5):
        v = np.log10(np.maximum(vol.astype(np.float32), 0) + 1.0)
        g = pv.ImageData()
        g.dimensions = np.array(v.shape) + 1
        g.cell_data['intensity'] = v.flatten(order='F')
        g = g.cell_data_to_point_data()
        return g.contour([float(v.max()) * frac], scalars='intensity')

    pl = pv.Plotter(notebook=True, shape=(1, 2), window_size=(1000, 480))
    pl.subplot(0, 0); pl.set_background('white'); pl.add_axes()
    pl.add_text('measured', font_size=10, color='black')
    pl.add_mesh(log_iso(data), color='royalblue', show_scalar_bar=False)
    pl.subplot(0, 1); pl.set_background('white'); pl.add_axes()
    pl.add_text('|FFT(image)|^2', font_size=10, color='black')
    pl.add_mesh(log_iso(recon), color='crimson', show_scalar_bar=False)
    pl.link_views(); pl.show()

**Read any tab's config** programmatically (keys: `instr`, `prep`, `data`, `rec`, `disp`):

In [ ]:
print('rec :', gui._tabs['rec'].get_config())
print('data:', gui._tabs['data'].get_config())